In [1]:
import xarray as xr
import pandas as pd
import numpy as np
import os

import importlib

import fates_calibration_library.oaat_functions as oaat

In [2]:
def get_global_data(ds, variables, param_dat):
    all_dat_list = []
    for variable in variables:
        mean_dat = ds[variable].sel(summation_var='mean').to_pandas().reset_index()
        mean_dat.columns = ['ensemble', 'mean_dat']
        
        iav_dat = ds[variable].sel(summation_var='iav').to_pandas().reset_index()
        iav_dat.columns = ['ensemble', 'iav_dat']
    
        dat = pd.merge(mean_dat, iav_dat, on='ensemble')
        
        default = dat[dat.ensemble == 0]
        default_mean = default['mean_dat'].values
        default_iav = default['iav_dat'].values
    
        dat['diff_mean'] = np.abs(dat['mean_dat'] - default_mean)
        dat['diff_iav'] = np.abs(dat['iav_dat'] - default_iav)
    
        dat['variable'] = variable
        all_dat_list.append(dat)
    
    all_dat = pd.concat(all_dat_list)

    dat_mean = all_dat.pivot(index=['ensemble'], columns='variable', values='mean_dat').reset_index()
    dat_mean.columns = [dat_mean.columns[0]] + [f"{c}_mean" for c in dat_mean.columns[1:]]
    
    diff_mean = all_dat.pivot(index=['ensemble'], columns='variable', values='diff_mean').reset_index()
    diff_mean.columns = [diff_mean.columns[0]] + [f"{c}_mean_diff" for c in diff_mean.columns[1:]]

    dat_iav = all_dat.pivot(index=['ensemble'], columns='variable', values='iav_dat').reset_index()
    dat_iav.columns = [dat_iav.columns[0]] + [f"{c}_iav" for c in dat_iav.columns[1:]]

    diff_iav = all_dat.pivot(index=['ensemble'], columns='variable', values='diff_iav').reset_index()
    diff_iav.columns = [diff_iav.columns[0]] + [f"{c}_iav_diff" for c in diff_iav.columns[1:]]

    diff_mean['mean_sum_diff'] = diff_mean.iloc[:, 1:].sum(axis=1)
    diff_iav['iav_sum_diff'] = diff_iav.iloc[:, 1:].sum(axis=1)

    dat = pd.merge(dat_mean, dat_iav, on='ensemble')
    dat = pd.merge(dat, diff_mean[['ensemble', 'mean_sum_diff']])
    dat = pd.merge(dat, diff_iav[['ensemble', 'iav_sum_diff']])
    dat = pd.merge(dat, param_dat, on='ensemble', how='outer')

    dat['sum_diff'] = dat.mean_sum_diff + dat.iav_sum_diff
    
    return dat

def calc_variance(default_value, min_value, max_value):
    return (default_value - min_value)**2 + (max_value - default_value)**2

def get_param_variance(parameters, variable, type, data):

    variance_dict = {}
    default = data[data.ensemble == 0].iloc[[0], :]
    variable_name = f"{variable}_{type}"
    
    for parameter in parameters:
        sub = data[data.parameter_name == parameter]
        sub_min = sub[sub.type == 'min']
        if len(sub_min) == 0:
            sub_min = default
        
        sub_max = sub[sub.type == 'max']
        if len(sub_max) == 0:
            sub_max = default

        variance = calc_variance(default[variable_name].values[0],
                                 sub_min[variable_name].values[0],
                                 sub_max[variable_name].values[0])

        variance_dict[parameter] = variance

    out_dat = pd.DataFrame.from_dict(variance_dict, orient='index')
    out_dat.columns = ['variance']
    out_dat['parameter'] = out_dat.index
    out_dat = out_dat.reset_index().drop(columns=['index'])
    out_dat['variable'] = variable_name
        
    return out_dat

def get_all_param_variances(parameters, variables, types, data):

    dat_list = []
    for variable in variables:
        for type in types:
            dat_list.append(get_param_variance(parameters, variable, type, data))

    dat_out = pd.concat(dat_list)

    return dat_out.pivot(index=['parameter'], columns='variable', values='variance').reset_index()

def get_min_max_diff(df, variables, types):
    parameters = np.unique(df['parameter_name'].dropna())
    default = df[df.ensemble == 0].iloc[[0], :]
    
    all_params_list = []
    for parameter in parameters:
        sub = df[df.parameter_name == parameter]
        
        sub_min = sub[sub.type == 'min']
        if len(sub_min) == 0:
            sub_min = default
        
        sub_max = sub[sub.type == 'max']
        if len(sub_max) == 0:
            sub_max = default
    
        all_vars_dict = {}
        for variable in variables:
            for type in types:
                variable_name = f"{variable}_{type}"
                diff = np.abs(sub_max[variable_name].values - sub_min[variable_name].values)
                all_vars_dict[variable_name] = diff
    
        all_vars_df = pd.DataFrame.from_dict(all_vars_dict, orient='index')
        all_vars_df.columns = ['diff']
        all_vars_df['variable'] = all_vars_df.index
        all_vars_df = all_vars_df.reset_index().drop(columns=['index'])
        all_vars_df['parameter_name'] = parameter
    
        all_params_list.append(all_vars_df)
    
    all_params = pd.concat(all_params_list)
    all_params_long = all_params.pivot(index=['parameter_name'], columns='variable',
                                       values='diff').reset_index()
    return all_params_long

def get_biome_data(ds, variables, param_dat):
    all_dat_list = []
    for variable in variables:
        mean_dat = ds[variable].sel(summation_var='mean').to_pandas().reset_index()
        mean_dat = mean_dat.melt(id_vars=['ensemble'], value_name='mean_dat')
        
        iav_dat = ds[variable].sel(summation_var='iav').to_pandas().reset_index()
        iav_dat = iav_dat.melt(id_vars=['ensemble'], value_name='iav_dat')
    
        dat = pd.merge(mean_dat, iav_dat, on=['ensemble', 'biome'])
        
        default = dat[dat.ensemble == 0].drop(columns=['ensemble'])
        default.columns = ['biome', 'default_mean', 'default_iav']

        dat = pd.merge(dat, default, on='biome')
    
        dat['diff_mean'] = np.abs(dat['mean_dat'] - dat['default_mean'])
        dat['diff_iav'] = np.abs(dat['iav_dat'] - dat['default_iav'])
    
        dat['variable'] = variable
        all_dat_list.append(dat)
    
    all_dat = pd.concat(all_dat_list).drop(columns=['default_mean', 'default_iav'])

    dat_mean = all_dat.pivot(index=['ensemble', 'biome'], columns='variable', values='mean_dat').reset_index()
    dat_mean.columns = np.append(dat_mean.columns[0:2], [f"{c}_mean" for c in dat_mean.columns[2:]])
    
    diff_mean = all_dat.pivot(index=['ensemble', 'biome'], columns='variable', values='diff_mean').reset_index()
    diff_mean.columns = np.append(diff_mean.columns[0:2], [f"{c}_diff_mean" for c in diff_mean.columns[2:]])

    dat_iav = all_dat.pivot(index=['ensemble', 'biome'], columns='variable', values='iav_dat').reset_index()
    dat_iav.columns = np.append(dat_iav.columns[0:2], [f"{c}_iav" for c in dat_iav.columns[2:]])

    diff_iav = all_dat.pivot(index=['ensemble', 'biome'], columns='variable', values='diff_iav').reset_index()
    diff_iav.columns = np.append(diff_iav.columns[0:2], [f"{c}_diff_iav" for c in diff_iav.columns[2:]])

    diff_mean['mean_sum_diff'] = diff_mean.iloc[:, 2:].sum(axis=1)
    diff_iav['iav_sum_diff'] = diff_iav.iloc[:, 2:].sum(axis=1)

    dat = pd.merge(dat_mean, dat_iav, on=['ensemble', 'biome'])
    dat = pd.merge(dat, diff_mean[['ensemble', 'biome', 'mean_sum_diff']], on=['ensemble', 'biome'])
    dat = pd.merge(dat, diff_iav[['ensemble', 'biome', 'iav_sum_diff']], on=['ensemble', 'biome'])
    dat = pd.merge(dat, param_dat, on='ensemble', how='outer')

    dat['sum_diff'] = dat.mean_sum_diff + dat.iav_sum_diff
    
    return dat

def get_min_max_diff_by_biome(df, variables, types):
    
    parameters = np.unique(df['parameter_name'].dropna())
    biomes = np.unique(df.biome)
    default = df[df.ensemble == 0]
    
    all_params_list = []
    for parameter in parameters:
        sub = df[df.parameter_name == parameter]
        
        sub_min = sub[sub.type == 'min']
        if len(sub_min) == 0:
            sub_min = default
        
        sub_max = sub[sub.type == 'max']
        if len(sub_max) == 0:
            sub_max = default
    
        all_vars_list = []
        for variable in variables:
            for type in types:
                variable_name = f"{variable}_{type}"
                diff = np.abs(sub_max[variable_name].values - sub_min[variable_name].values)
                var_dat = pd.DataFrame({"diff": diff})
                var_dat['variable'] = variable_name
                var_dat['biome'] = sub_max['biome'].values
                all_vars_list.append(var_dat)
    
        all_vars_df = pd.concat(all_vars_list)
        all_vars_df['parameter_name'] = parameter
    
        all_params_list.append(all_vars_df)
    
    all_params = pd.concat(all_params_list)
    all_params_long = all_params.pivot(index=['parameter_name', 'biome'], columns='variable',
                                       values='diff').reset_index()
    return all_params_long

def get_zonal_dat(ds, variables, param_info):
    all_vars_list = []
    for variable in variables:
        dat = ds[variable].to_dataframe(name='mean_dat').reset_index()
        dat['variable'] = variable
        all_vars_list.append(dat)
    all_dat = pd.concat(all_vars_list)
    all_dat.columns = ['ensemble', 'latitude', 'mean_dat', 'variable']
    dat_out = all_dat.pivot(index=['ensemble', 'latitude'], columns='variable', values='mean_dat').reset_index()

    dat_out = pd.merge(dat_out, param_info, on='ensemble', how='outer')

    return dat_out

def get_climatology_dat(ds, variables, param_info):
    all_vars_list = []
    for variable in variables:
        dat = ds[variable].to_dataframe(name='mean_dat').reset_index()
        dat['variable'] = variable
        all_vars_list.append(dat)
    all_dat = pd.concat(all_vars_list)
    all_dat.columns = ['ensemble', 'month', 'mean_dat', 'variable']
    dat_out = all_dat.pivot(index=['ensemble', 'month'], columns='variable', values='mean_dat').reset_index()

    dat_out = pd.merge(dat_out, param_info, on='ensemble', how='outer')

    return dat_out

def get_var_diff(df, param_info, variables, types, relative=True):
    default = df[df.ensemble == 0].iloc[[0], :]
    parameters = np.unique(df['parameter_name'].dropna())
    
    all_vars_list = []
    for variable in variables:
        for type in types:
            variable_name = f"{variable}_{type}"

            pars_list = []
            for parameter in parameters:
                sub = df[df.parameter_name == parameter].copy()
                
                sub_min = sub[sub.type == 'min'].copy()
                if len(sub_min) == 0:
                    sub_min = default
                
                sub_max = sub[sub.type == 'max'].copy()
                if len(sub_max) == 0:
                    sub_max = default
    
                var_diff = np.abs(sub_max[variable_name].values[0] - sub_min[variable_name].values[0])
                if relative:
                    diff = (var_diff/default[variable_name].values[0])*100.0
                else:
                    diff = var_diff
                
                dat = pd.DataFrame({'diff': [diff]})
                dat['parameter_name'] = parameter
                pars_list.append(dat)
            pars_df = pd.concat(pars_list)
            pars_df['variable'] = variable_name
            all_vars_list.append(pars_df)
    
    all_vars = pd.concat(all_vars_list)
    dat_out = all_vars.pivot(index=['parameter_name'], columns='variable', values='diff').reset_index()
    dat_out = pd.merge(dat_out, param_info, on='parameter_name')
    
    return dat_out

def get_var_diff_by_biome(df, param_info, variables, types, relative=True):
    
    biomes = np.unique(df.biome)

    all_dat_list = []
    for biome in biomes:
        sub = df[df.biome == biome]
        dat_diff = get_var_diff(sub, param_info, variables, types, relative=relative)
        dat_diff['biome'] = biome
        all_dat_list.append(dat_diff)

    return pd.concat(all_dat_list)

def update_file(indir, nc_file, ensembles):
    file = os.path.join(indir, nc_file)
    dat = xr.open_dataset(file)
    return dat.where(dat.ensemble.isin(ensembles), drop=True)

def write_parameters(file_path, param_list):
    formatted_list = [item + "\n" for item in param_list]

    with open(file_path, 'w') as file:
        file.writelines(formatted_list)

In [3]:
VARIABLES = ['GPP', 'EFLX_LH_TOT', 'FSH', 'EF', 'SOILWATER_10CM',
             'FSR', 'FSA', 'FIRE', 'RLNS', 'TV', 'ASA', 'RN', 'BTRANMN',
             'QRUNOFF', 'QSOIL', 'QVEGT', 'QVEGE']
TYPES = ['mean', 'iav']

# File Names

In [4]:
parameter_dir = '/glade/work/afoster/FATES_calibration/parameter_files'
hist_dir = '/glade/work/afoster/FATES_calibration/history_files/compiled_files'
out_dir = '/glade/work/afoster/FATES_calibration/history_files/shiny_data'

fates_key_file = os.path.join(parameter_dir, 'fates_oaat', 'fates_oaat_key.csv')
fates_oaat_file = os.path.join(parameter_dir, 'param_list_sci.1.85.1_api.40.0.0_updates.xls')
clm_param_list = os.path.join(parameter_dir, 'CLM5_Parameter_info.csv')
clm_key_file = os.path.join(parameter_dir, 'clm6sp_oaat_key.csv')

fates_glob_file = os.path.join(hist_dir, 'fates_oaat_area_means.nc')
fates_biome_file = os.path.join(hist_dir, 'fates_oaat_biome_area_means.nc')

clmfates_glob_file = os.path.join(hist_dir, 'fates_oaat_clmpars_area_means.nc')
clmfates_biome_file = os.path.join(hist_dir, 'fates_oaat_clmpars_biome_area_means.nc')

clm_glob_file = os.path.join(hist_dir, 'clm_oaat_area_means.nc')
clm_biome_file = os.path.join(hist_dir, 'clm_oaat_biome_area_means.nc')

fates_zonal_file = os.path.join(hist_dir, 'fates_oaat_zonal_means.nc')
fatesclm_zonal_file = os.path.join(hist_dir, 'fates_oaat_clmpars_zonal_means.nc')
clm_zonal_file = os.path.join(hist_dir, 'clm_oaat_zonal_means.nc')

fates_clim_file = os.path.join(hist_dir, 'fates_oaat_climatology.nc')
fatesclm_clim_file = os.path.join(hist_dir, 'fates_oaat_clmpars_climatology.nc')
clm_clim_file = os.path.join(hist_dir, 'clm_oaat_climatology.nc')

# Parameter Information

In [5]:
fates_key = pd.read_csv(fates_key_file, index_col=[0])
fates_param_dat = oaat.get_fates_param_dat(fates_oaat_file, fates_key, to_xarray=False)
fates_param_dat['model'] = 'FATES'
fates_param_dat_reduced = fates_param_dat.loc[:, ['parameter_name', 'category', 'subcategory', 'long_name', 'model']]

clm_key = pd.read_csv(clm_key_file, header=None)
clm_key.columns = ['ensemble', 'parameter_name', 'type']
clm_param_dat = oaat.get_clm_param_dat(clm_param_list, clm_key, to_xarray=False)
clm_param_dat['model'] = 'CLM'
clm_param_dat_reduced = clm_param_dat.loc[:, ['parameter_name', 'category', 'subcategory', 'long_name', 'model']]

all_params = pd.concat([fates_param_dat_reduced, clm_param_dat_reduced]).drop_duplicates()

# Global annual outputs

In [6]:
# open global annual outputs
fates_glob_dat = xr.open_dataset(fates_glob_file)
fatesclm_glob_dat = xr.open_dataset(clmfates_glob_file)
clm_glob_dat = xr.open_dataset(clm_glob_file)

In [7]:
# get global values
fates_fates_glob = get_global_data(fates_glob_dat, VARIABLES, fates_param_dat)
fatesclm_glob = get_global_data(fatesclm_glob_dat, VARIABLES, clm_param_dat)

fates_glob = pd.concat([fates_fates_glob, fatesclm_glob])
fates_glob['model'] = 'FATES'

clm_glob = get_global_data(clm_glob_dat, VARIABLES, clm_param_dat)
clm_glob['model'] = 'CLM'

# combine datasets
fates_clm_global_annual_data = pd.concat([clm_glob, fates_glob])

# Non-Zero Parameters

In [8]:
# find non-zero parameters
fates_fates_nonzero = np.unique(fates_fates_glob[fates_fates_glob.sum_diff > 0.0].copy().parameter_name.values)
fates_clm_nonzero = np.unique(fatesclm_glob[fatesclm_glob.sum_diff > 0.0].copy().parameter_name.values)
clm_nonzero = np.unique(clm_glob[clm_glob.sum_diff > 0.0].copy().parameter_name.values)

all_nonzero = np.unique(np.append(np.append(fates_fates_nonzero, fates_clm_nonzero), 
                                  clm_nonzero))
fates_only_parameters = np.append([param for param in fates_clm_nonzero if param not in clm_nonzero],
                                  fates_fates_nonzero)
SPECIAL_PARS = ['fates_vcmaxse_clmdef', 'fates_vcmaxhd_clmdef',
                'fates_vcmaxha_clmdef', 'fates_jmaxse_clmdef',
                'fates_jmaxhd_clmdef', 'fates_jmaxha_clmdef',
                'fates_new_def']
fates_only_parameters = [f for f in fates_only_parameters if f not in SPECIAL_PARS]
clm_only_parameters = [param for param in clm_nonzero if param not in fates_clm_nonzero]
common_parameters = [param for param in clm_nonzero if param in fates_clm_nonzero]

clm_pars = np.concatenate([clm_only_parameters, common_parameters])
fates_pars = np.concatenate([fates_only_parameters, common_parameters])

In [9]:
# find keys associated with non-zero parameters
clm_key['key'] = [int(ens.strip()[-4:]) for ens in clm_key.ensemble]

fates_only_ens = np.append(fates_key[fates_key.parameter_name.isin(fates_only_parameters)].ensemble.values, 0)
clm_only_ens = np.append(clm_key[clm_key.parameter_name.isin(clm_pars)].key, 0)
fates_clm_ens = np.append(clm_key[clm_key.parameter_name.isin(fates_pars)].key, 0)

## Ensemble Variance

In [10]:
fates_variance = get_all_param_variances(fates_pars, VARIABLES, TYPES, fates_glob)
fates_variance['model'] = 'FATES'
fates_variance = fates_variance[fates_variance['parameter'].isin(fates_pars)]


clm_variance = get_all_param_variances(clm_pars, VARIABLES, TYPES, clm_glob)
clm_variance['model'] = 'CLM'
clm_variance = clm_variance[clm_variance['parameter'].isin(clm_pars)]

fates_clm_variance = pd.concat([fates_variance, clm_variance])

## Min - Max Differences

In [11]:
fates_min_max = get_min_max_diff(fates_glob, VARIABLES, TYPES)
fates_min_max['model'] = 'FATES'
fates_min_max = fates_min_max[fates_min_max['parameter_name'].isin(fates_pars)]

clm_min_max = get_min_max_diff(clm_glob, VARIABLES, TYPES)
clm_min_max['model'] = 'CLM'
clm_min_max = clm_min_max[clm_min_max['parameter_name'].isin(clm_pars)]

fates_clm_min_max = pd.concat([fates_min_max, clm_min_max])

## Variable Differences

In [12]:
clm_var_diff = get_var_diff(clm_glob, all_params, VARIABLES, TYPES)
clm_var_diff['model'] = 'CLM'
clm_var_diff = clm_var_diff[clm_var_diff['parameter_name'].isin(clm_pars)]

fates_var_diff = get_var_diff(fates_glob, all_params, VARIABLES, TYPES)
fates_var_diff['model'] = 'FATES'
fates_var_diff = fates_var_diff[fates_var_diff['parameter_name'].isin(fates_pars)]


all_clm_var_diff = pd.concat([clm_var_diff, fates_var_diff])

# Biome Values

In [13]:
# open biome datasets
fates_biome_dat = xr.open_dataset(fates_biome_file)
fatesclm_biome_dat = xr.open_dataset(clmfates_biome_file)
clm_biome_dat = xr.open_dataset(clm_biome_file)

In [14]:
# get biome values
fates_biome = get_biome_data(fates_biome_dat, VARIABLES, fates_param_dat)
fates_biome['model'] = 'FATES'

fatesclm_biome = get_biome_data(fatesclm_biome_dat, VARIABLES, clm_param_dat)
fatesclm_biome['model'] = 'FATES'

clm_biome = get_biome_data(clm_biome_dat, VARIABLES, clm_param_dat)
clm_biome['model'] = 'CLM'

fates_clm_biome = pd.concat([clm_biome, fates_biome, fatesclm_biome[fatesclm_biome.ensemble != 0].copy()])

In [15]:
# max - min for biome df
fates_min_max_biome = get_min_max_diff_by_biome(fates_biome, VARIABLES, TYPES)
fates_min_max_biome['model'] = 'FATES'
fates_min_max_biome = fates_min_max_biome[fates_min_max_biome['parameter_name'].isin(fates_pars)]

fatesclm_min_max_biome = get_min_max_diff_by_biome(fatesclm_biome, VARIABLES, TYPES)
fatesclm_min_max_biome['model'] = 'FATES'
fatesclm_min_max_biome = fatesclm_min_max_biome[fatesclm_min_max_biome['parameter_name'].isin(fates_pars)]

clm_min_max_biome = get_min_max_diff_by_biome(clm_biome, VARIABLES, TYPES)
clm_min_max_biome['model'] = 'CLM'
clm_min_max_biome = clm_min_max_biome[clm_min_max_biome['parameter_name'].isin(clm_pars)]

fates_clm_min_max_biome = pd.concat([fates_min_max_biome,
                                     fatesclm_min_max_biome,
                                     clm_min_max_biome])

## Variable Differences

In [16]:
clm_biome_var_diff = get_var_diff_by_biome(clm_biome, all_params, VARIABLES, TYPES)
clm_biome_var_diff['model'] = 'CLM'
clm_biome_var_diff = clm_biome_var_diff[clm_biome_var_diff['parameter_name'].isin(clm_pars)]

fatesclm_biome_var_diff = get_var_diff_by_biome(fatesclm_biome, all_params, VARIABLES, TYPES)
fatesclm_biome_var_diff['model'] = 'FATES'
fatesclm_biome_var_diff = fatesclm_biome_var_diff[fatesclm_biome_var_diff['parameter_name'].isin(fates_pars)]

fates_biome_var_diff = get_var_diff_by_biome(fates_biome, all_params, VARIABLES, TYPES)
fates_biome_var_diff['model'] = 'FATES'
fates_biome_var_diff = fates_biome_var_diff[fates_biome_var_diff['parameter_name'].isin(fates_pars)]

In [17]:
all_biome_var_diff = pd.concat([clm_biome_var_diff,
                                fatesclm_biome_var_diff,
                                fates_biome_var_diff])

In [18]:
fates_biome = fates_biome[fates_biome['parameter_name'].isin(fates_pars)]
fatesclm_biome = fatesclm_biome[fatesclm_biome['parameter_name'].isin(fates_pars)]
clm_biome = clm_biome[clm_biome['parameter_name'].isin(clm_pars)]

# Zonal Data

In [19]:
# get zonal data
fates_zonal = xr.open_dataset(fates_zonal_file)
fatesclm_zonal = xr.open_dataset(fatesclm_zonal_file)
clm_zonal = xr.open_dataset(clm_zonal_file)

In [20]:
fates_zonal_dat = get_zonal_dat(fates_zonal, VARIABLES, fates_param_dat)
fates_zonal_dat['model'] = 'FATES'
fates_zonal_dat = fates_zonal_dat[fates_zonal_dat['parameter_name'].isin(fates_pars)]

fatesclm_zonal_dat = get_zonal_dat(fatesclm_zonal, VARIABLES, clm_param_dat)
fatesclm_zonal_dat['model'] = 'FATES'
fatesclm_zonal_dat = fatesclm_zonal_dat[fatesclm_zonal_dat['parameter_name'].isin(fates_pars)]

clm_zonal_dat = get_zonal_dat(clm_zonal, VARIABLES, clm_param_dat)
clm_zonal_dat['model'] = 'CLM'
clm_zonal_dat = clm_zonal_dat[clm_zonal_dat['parameter_name'].isin(clm_pars)]

all_zonal = pd.concat([fates_zonal_dat,
                       fatesclm_zonal_dat,
                       clm_zonal_dat])

# Climatology

In [21]:
# get climatology data
fates_clim = xr.open_dataset(fates_clim_file)
fatesclm_clim = xr.open_dataset(fatesclm_clim_file)
clm_clim = xr.open_dataset(clm_clim_file)

In [22]:
fates_clim_dat = get_climatology_dat(fates_clim, VARIABLES, fates_param_dat)
fates_clim_dat['model'] = 'FATES'
fates_clim_dat = fates_clim_dat[fates_clim_dat['parameter_name'].isin(fates_pars)]

fatesclm_clim_dat = get_climatology_dat(fatesclm_clim, VARIABLES, clm_param_dat)
fatesclm_clim_dat['model'] = 'FATES'
fatesclm_clim_dat = fatesclm_clim_dat[fatesclm_clim_dat['parameter_name'].isin(fates_pars)]

clm_clim_dat = get_climatology_dat(clm_clim, VARIABLES, clm_param_dat)
clm_clim_dat['model'] = 'CLM'
clm_clim_dat = clm_clim_dat[clm_clim_dat['parameter_name'].isin(clm_pars)]

all_clim = pd.concat([fates_clim_dat, fatesclm_clim_dat, clm_clim_dat])

## Subset Maps

In [52]:
vars_to_drop = ['RH2M', 'Precip', 'LAI', 'FSDS', 'Temp']

# subset out parameters that have zero effect
fates_update = update_file(hist_dir, 'fates_oaat_annual_maps.nc', fates_only_ens).drop_vars(vars_to_drop)
clm_update = update_file(hist_dir, 'clm_oaat_annual_maps.nc', clm_only_ens).drop_vars(vars_to_drop)
fatesclm_update = update_file(hist_dir, 'fates_oaat_clmpars_annual_maps.nc', fates_clm_ens).drop_vars(vars_to_drop)

# Write to files

In [53]:
# maps
fates_update.to_netcdf(os.path.join(out_dir, 'fates_oaat_annual_maps_nonzero.nc'))
clm_update.to_netcdf(os.path.join(out_dir, 'clm_oaat_annual_maps_nonzero.nc'))
fatesclm_update.to_netcdf(os.path.join(out_dir, 'fates_oaat_clmpars_annual_maps_nonzero.nc'))

In [23]:
# csvs
fates_clm_global_annual_data.to_csv(os.path.join(out_dir, 'fates_clm_global_annual_data.csv'))
fates_clm_variance.to_csv(os.path.join(out_dir, 'fates_clm_global_annual_variance.csv'))
fates_clm_min_max.to_csv(os.path.join(out_dir, 'fates_clm_min_max_global.csv'))
all_clm_var_diff.to_csv(os.path.join(out_dir, 'fates_clm_var_diff_data.csv'))
fates_clm_biome.to_csv(os.path.join(out_dir, 'fates_clm_biome_annual_data.csv'))
fates_clm_min_max_biome.to_csv(os.path.join(out_dir, 'fates_clm_min_max_biome.csv'))
all_biome_var_diff.to_csv(os.path.join(out_dir, 'fates_clm_biome_var_diff_data.csv'))
all_zonal.to_csv(os.path.join(out_dir, 'fates_clm_zonal_data.csv'))
all_clim.to_csv(os.path.join(out_dir, 'fates_clm_clim_data.csv'))

In [24]:
# parameter information
all_params.to_csv(os.path.join(out_dir, 'parameter_info.csv'))

write_parameters(os.path.join(out_dir, 'fates_only.txt'), fates_only_parameters)
write_parameters(os.path.join(out_dir, 'clm_only.txt'), clm_only_parameters)
write_parameters(os.path.join(out_dir, 'fates_and_clm.txt'), common_parameters)